# Train Target Over Time Per Market

Loads `data/train.csv` and plots target over time for each market.

In [ ]:
from pathlib import Path
import math

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DATA_PATH = Path('../data/train.csv')

df = pd.read_csv(DATA_PATH)
df['delivery_start'] = pd.to_datetime(df['delivery_start'], errors='coerce')
df = df.dropna(subset=['delivery_start']).copy()
df['market'] = df['market'].astype(str)
df = df.sort_values(['market', 'delivery_start']).reset_index(drop=True)

markets = sorted(df['market'].unique())
print(f'Rows: {len(df):,}')
print(f'Markets: {markets}')
print(f'Date range: {df["delivery_start"].min()} -> {df["delivery_start"].max()}')

In [ ]:
if not markets:
    print('No market data found.')
else:
    cols = 2
    rows = math.ceil(len(markets) / cols)
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f'Market {m}' for m in markets],
        vertical_spacing=0.08,
    )

    for i, m in enumerate(markets, start=1):
        r = (i - 1) // cols + 1
        c = (i - 1) % cols + 1
        s = df[df['market'] == m]
        fig.add_trace(
            go.Scatter(
                x=s['delivery_start'],
                y=s['target'],
                mode='lines',
                line=dict(width=1.1, color='#1f77b4'),
                opacity=0.6,
                showlegend=False,
                name=f'Market {m}',
                hovertemplate='delivery_start=%{x}<br>target=%{y:.3f}<extra>Market ' + m + '</extra>',
            ),
            row=r,
            col=c,
        )

    fig.update_layout(
        title='Train target over time per market (raw)',
        template='plotly_white',
        height=max(500, 300 * rows),
    )
    fig.update_xaxes(title_text='delivery_start')
    fig.update_yaxes(title_text='target')
    fig.show()

In [ ]:
daily = (
    df.assign(delivery_day=df['delivery_start'].dt.floor('D'))
    .groupby(['market', 'delivery_day'], as_index=False)['target']
    .mean()
    .rename(columns={'delivery_day': 'delivery_start', 'target': 'target_daily_mean'})
)

if not markets:
    print('No market data found.')
else:
    cols = 2
    rows = math.ceil(len(markets) / cols)
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f'Market {m}' for m in markets],
        vertical_spacing=0.08,
    )

    for i, m in enumerate(markets, start=1):
        r = (i - 1) // cols + 1
        c = (i - 1) % cols + 1
        s = daily[daily['market'] == m]
        fig.add_trace(
            go.Scatter(
                x=s['delivery_start'],
                y=s['target_daily_mean'],
                mode='lines',
                line=dict(width=1.8, color='#d62728'),
                showlegend=False,
                name=f'Market {m} daily mean',
                hovertemplate='delivery_day=%{x}<br>target_daily_mean=%{y:.3f}<extra>Market ' + m + '</extra>',
            ),
            row=r,
            col=c,
        )

    fig.update_layout(
        title='Train target over time per market (daily mean)',
        template='plotly_white',
        height=max(500, 300 * rows),
    )
    fig.update_xaxes(title_text='delivery_day')
    fig.update_yaxes(title_text='target_daily_mean')
    fig.show()